# 🎯 OneVoice Edge — Target ASR Benchmark (GIPFormer & SenseVoice)
**Models Evaluated:** GIPFormer ASR (g-group-ai-lab/gipformer-65M-rnnt INT8) & SenseVoice Small (iic/SenseVoiceSmall)
**Features:** Clean vs Noisy Audio Evaluation · Noise-Type & SNR Breakdown · Industrial Construction Benchmark

## Cell 1 — Mount Drive & Install Project Dependencies

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
    MODEL_OUTPUT = '/content/drive/MyDrive/onevoice_models/asr_eval'
else:
    DATASET_ROOT = './onevoice_audio_v1'
    MODEL_OUTPUT = './models/asr_eval'

os.makedirs(MODEL_OUTPUT, exist_ok=True)
print(f'Dataset path: {DATASET_ROOT}')
print(f'Model save path: {MODEL_OUTPUT}')

# Install dependencies for sherpa-onnx & SenseVoice
!pip install -q sherpa-onnx funasr modelscope funasr_onnx jiwer torchaudio soundfile librosa pandas tqdm huggingface_hub

## Cell 2 — Clone Project Repository

In [ ]:
import os, sys

if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned. Pulling latest...')
    !cd /content/OneVoice && git pull

for p in ['onevoice-edge/src', 'src']:
    full = os.path.join('/content/OneVoice', p)
    if os.path.exists(full):
        sys.path.append(full)
        break
print('✅ Project src path linked.')

## Cell 3 — Load Manifest & Prepare Evaluation Sets

In [ ]:
import os, json, pandas as pd

MANIFEST_PATH = os.path.join(DATASET_ROOT, 'manifest.jsonl')
CLEAN_DIR = os.path.join(DATASET_ROOT, 'clean')
NOISY_DIR = os.path.join(DATASET_ROOT, 'noisy')

assert os.path.exists(MANIFEST_PATH), f'Manifest not found at {MANIFEST_PATH}! Please check Cell 1.'

entries = []
with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            entries.append(json.loads(line.strip()))

df_manifest = pd.DataFrame(entries)
print(f'Total manifest samples loaded: {len(df_manifest)}')
print(df_manifest[['audio', 'text', 'noise_type', 'snr_db', 'split']].head())

## Cell 4 — Benchmark GIPFormer ASR (Vietnamese Edge Model via sherpa-onnx)

In [ ]:
import re, torch, jiwer, librosa, numpy as np
from tqdm.notebook import tqdm
from huggingface_hub import hf_hub_download
import sherpa_onnx

print('⚙️ Loading GIPFormer ASR (g-group-ai-lab/gipformer-65M-rnnt INT8)...')

repo = 'g-group-ai-lab/gipformer-65M-rnnt'
encoder_p = hf_hub_download(repo_id=repo, filename='encoder-epoch-35-avg-6.int8.onnx')
decoder_p = hf_hub_download(repo_id=repo, filename='decoder-epoch-35-avg-6.int8.onnx')
joiner_p  = hf_hub_download(repo_id=repo, filename='joiner-epoch-35-avg-6.int8.onnx')
tokens_p  = hf_hub_download(repo_id=repo, filename='tokens.txt')

recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=encoder_p,
    decoder=decoder_p,
    joiner=joiner_p,
    tokens=tokens_p,
    num_threads=4,
    sample_rate=16000,
    feature_dim=80,
    decoding_method='greedy_search'
)
print('✅ GIPFormer ASR loaded successfully!')

def clean_text(t):
    return ' '.join(re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', str(t).lower()).split())

def transcribe_gipformer(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    stream = recognizer.create_stream()
    stream.accept_waveform(16000, audio.astype(np.float32))
    recognizer.decode_streams([stream])
    return clean_text(stream.result.text)

eval_samples = df_manifest.head(300)
results_gip = []

print(f'🚀 Benchmarking GIPFormer ASR on {len(eval_samples)} construction audio samples...')
for _, row in tqdm(eval_samples.iterrows(), total=len(eval_samples)):
    cp = os.path.join(CLEAN_DIR, row['clean_audio'])
    np_p = os.path.join(NOISY_DIR, row['audio'])
    if not os.path.exists(np_p): continue
    
    gt = clean_text(row['text'])
    pred_clean = transcribe_gipformer(cp) if os.path.exists(cp) else ''
    pred_noisy = transcribe_gipformer(np_p)
    
    wer_clean = jiwer.wer(gt, pred_clean) * 100 if gt and pred_clean else 0
    wer_noisy = jiwer.wer(gt, pred_noisy) * 100 if gt and pred_noisy else 0
    
    results_gip.append({
        'audio': row['audio'],
        'gt': gt,
        'pred_clean': pred_clean,
        'pred_noisy': pred_noisy,
        'wer_clean': wer_clean,
        'wer_noisy': wer_noisy,
        'noise_type': row.get('noise_type', 'unknown'),
        'snr_db': row.get('snr_db', 0)
    })

df_gip = pd.DataFrame(results_gip)
print('\n' + '='*55)
print('📊 GIPFormer ASR (INT8 Edge Model) PERFORMANCE SUMMARY:')
print(f'  • Mean WER (Clean Audio): {df_gip["wer_clean"].mean():.2f}%')
print(f'  • Mean WER (Noisy Audio): {df_gip["wer_noisy"].mean():.2f}%')
print(f'  • Noise Impact Gap      : +{df_gip["wer_noisy"].mean() - df_gip["wer_clean"].mean():.2f}% WER')
print('='*55)

## Cell 5 — Benchmark SenseVoiceSmall Model (English / Multi-lang ASR)

In [ ]:
from funasr import AutoModel

print('⚙️ Loading SenseVoiceSmall model (iic/SenseVoiceSmall)...')
sv_model = AutoModel(
    model='iic/SenseVoiceSmall',
    vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    vad_kwargs={'max_single_segment_time': 30000},
    device='cuda' if torch.cuda.is_available() else 'cpu',
    disable_update=True
)
print('✅ SenseVoiceSmall loaded!')

results_sv = []
for _, row in tqdm(eval_samples.iterrows(), total=len(eval_samples)):
    np_p = os.path.join(NOISY_DIR, row['audio'])
    if not os.path.exists(np_p): continue
    gt = clean_text(row['text'])
    res = sv_model.generate(input=np_p, cache={}, language='auto', use_itn=True)
    pred_text = clean_text(res[0]['text']) if res and len(res) > 0 else ''
    results_sv.append({
        'audio': row['audio'],
        'gt': gt,
        'pred': pred_text,
        'wer': jiwer.wer(gt, pred_text) * 100 if gt and pred_text else 0,
        'noise_type': row.get('noise_type', 'unknown'),
        'snr_db': row.get('snr_db', 0)
    })

df_sv = pd.DataFrame(results_sv)
print('\n' + '='*55)
print('📊 SenseVoiceSmall PERFORMANCE SUMMARY:')
print(f'  • Mean WER (Noisy Audio): {df_sv["wer"].mean():.2f}%')
print('='*55)

## Cell 6 — Noise-Type & SNR Breakdown Report

In [ ]:
print('='*60)
print('📊 GIPFormer WER BREAKDOWN BY NOISE TYPE & SNR LEVEL')
print('='*60)
print('\nBy Noise Type:')
print(df_gip.groupby('noise_type')['wer_noisy'].mean().round(2))
print('\nBy SNR Level (dB):')
print(df_gip.groupby('snr_db')['wer_noisy'].mean().round(2))
print('='*60)

# Save evaluation report to Drive
eval_csv = os.path.join(MODEL_OUTPUT, 'gipformer_benchmark_results.csv')
df_gip.to_csv(eval_csv, index=False)
print(f'💾 Evaluation report saved to: {eval_csv}')